# README

The following code is partly based on Andrej Karpathy
's minimal PyTorch re-implementation of the OpenAI GPT. Please check out the original codebase via https://github.com/karpathy/minGPT

Authors: Kasper Liu (main contributor), Julia Grzeskowiak

# Installing required dependencies

In [ ]:
%pip install torch==2.9.1 datasets==4.0.0 regex==2024.11.6 matplotlib==3.10.0 scikit-learn==1.6.1 seaborn==0.13.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.3/39.3 MB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.0/90.0 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.5

# BPETokenizer - Slightly-modified from karpathy/minGPT

Karpathy: BPE is short for Byte Pair Encoder. It translates arbitrary utf-8 strings into
sequences of integers, where each integer represents small chunks of commonly
occuring characters. This implementation is based on openai's gpt2 encoder.py:
https://github.com/openai/gpt-2/blob/master/src/encoder.py
but was mildly modified because the original implementation is a bit confusing.
I also tried to add as many comments as possible, my own understanding of what's
going on.

In [ ]:
import os
import json
import regex as re
import requests

import torch

# -----------------------------------------------------------------------------

def bytes_to_unicode():
    """
    Every possible byte (really an integer 0..255) gets mapped by OpenAI to a unicode
    character that represents it visually. Some bytes have their appearance preserved
    because they don't cause any trouble. These are defined in list bs. For example:
    chr(33) returns "!", so in the returned dictionary we simply have d[33] -> "!".
    However, chr(0), for example, is '\x00', which looks ugly. So OpenAI maps these
    bytes, into new characters in a range where chr() returns a single nice character.
    So in the final dictionary we have d[0] -> 'Ā' instead, which is just chr(0 + 2**8).
    In particular, the space character is 32, which we can see by ord(' '). Instead,
    this function will shift space (32) by 256 to 288, so d[32] -> 'Ġ'.
    So this is just a simple one-to-one mapping of bytes 0..255 into unicode characters
    that "look nice", either in their original form, or a funny shifted character
    like 'Ā', or 'Ġ', etc.
    """
    # the 188 integers that render fine in their original form and need no shifting
    bs = list(range(ord("!"), ord("~")+1))+list(range(ord("¡"), ord("¬")+1))+list(range(ord("®"), ord("ÿ")+1))
    cs = bs[:] # all integers b in bs will simply map to chr(b) in the output dict
    # now get the representations of the other 68 integers that do need shifting
    # each will get mapped chr(256 + n), where n will grow from 0...67 in the loop
    n = 0
    for b in range(2**8):
        if b not in bs:
            # if this byte is "ugly" then map it to the next available "nice" character
            bs.append(b)
            cs.append(2**8+n)
            n += 1
    cs = [chr(n) for n in cs]
    d = dict(zip(bs, cs))
    return d

def get_pairs(word):
    """
    Return all bigrams as a set of tuples, of consecutive elements in the iterable word.
    """
    pairs = set()
    prev_char = word[0]
    for char in word[1:]:
        pairs.add((prev_char, char))
        prev_char = char
    return pairs

class Encoder:

    def __init__(self, encoder, bpe_merges):
        # byte encoder/decoder
        self.byte_encoder = bytes_to_unicode()
        self.byte_decoder = {v:k for k, v in self.byte_encoder.items()}
        # bpe token encoder/decoder
        self.encoder = encoder
        self.decoder = {v:k for k,v in self.encoder.items()}
        # bpe merge list that defines the bpe "tree", of tuples (a,b) that are to merge to token ab
        self.bpe_ranks = dict(zip(bpe_merges, range(len(bpe_merges))))
        # the splitting pattern used for pre-tokenization
        # Should haved added re.IGNORECASE so BPE merges can happen for capitalized versions of contractions <-- original openai comment
        """
        ok so what is this regex looking for, exactly?
        python re reference: https://docs.python.org/3/library/re.html
        - the vertical bars | is OR, so re.findall will chunkate text as the pieces match, from left to right
        - '\\'s' would split up things like Andrej's -> (Andrej, 's)
        - ' ?\\p{L}': optional space followed by 1+ unicode code points in the category "letter"
        - ' ?\\p{N}': optional space followed by 1+ unicode code points in the category "number"
        - ' ?[^\\s\\p{L}\\p{N}]+': optional space, then 1+ things that are NOT a whitespace, letter or number
        - '\\s+(?!\\S)': 1+ whitespace characters (e.g. space or tab or etc) UNLESS they are followed by non-whitespace
                       so this will consume whitespace characters in a sequence but exclude the last whitespace in
                       that sequence. that last whitespace has the opportunity to then match the optional ' ?' in
                       earlier patterns.
        - '\\s+': 1+ whitespace characters, intended probably to catch a full trailing sequence of whitespaces at end of string
        So TLDR:
        - we are special casing a few common apostrophe constructs ('s, 't, 're, ...) and making those into separate tokens
        - we then separate out strings into consecutive chunks of 1) letters, 2) numbers, 3) non-letter-numbers, 4) whitespaces
        """
        self.pat = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")
        self.cache = {}

    def bpe(self, token):
        """
        this function uses self.bpe_ranks to iteratively merge all the possible bpe tokens
        up the tree. token is a string of one individual 'word' (after regex tokenization)
        and after byte encoding, e.g. 'Ġthere'.
        """
        # token is a string of one individual 'word', after byte encoding, e.g. 'Ġthere'

        # memoization, for efficiency
        if token in self.cache:
            return self.cache[token]

        word = tuple(token) # individual characters that make up the token, in a tuple
        pairs = get_pairs(word) # get all bigrams

        if not pairs:
            return token

        while True:

            # find the next lowest rank bigram that can be merged
            bigram = min(pairs, key = lambda pair: self.bpe_ranks.get(pair, float('inf')))
            if bigram not in self.bpe_ranks:
                break # no more bigrams are eligible to be merged
            first, second = bigram

            # we will now replace all occurences of (first, second) in the list of current
            # words into one merged token first_second, in the output list new_words
            new_word = []
            i = 0
            while i < len(word):

                # find the next occurence of first in the sequence of current words
                try:
                    j = word.index(first, i)
                    new_word.extend(word[i:j])
                    i = j
                except:
                    new_word.extend(word[i:])
                    break

                # if this occurence is also followed by second, then merge them into one
                if word[i] == first and i < len(word)-1 and word[i+1] == second:
                    new_word.append(first+second)
                    i += 2
                else:
                    new_word.append(word[i])
                    i += 1

            # all occurences of (first, second) have been merged to first_second
            new_word = tuple(new_word)
            word = new_word
            if len(word) == 1:
                break
            else:
                pairs = get_pairs(word)

        # concat all words into a string, and use ' ' as the separator. Note that
        # by now all characters have been byte encoded, guaranteeing that ' ' is
        # not used in the actual data and is a 'special' delimiter character
        word = ' '.join(word)

        # cache the result and return
        self.cache[token] = word
        return word

    def encode(self, text):
        """ string goes in, list of integers comes out """
        bpe_idx = []
        # pre-tokenize the input text into string tokens (words, roughly speaking)
        tokens = re.findall(self.pat, text)
        # process each token into BPE integers
        for token in tokens:
            # encode the token as a bytes (b'') object
            token_bytes = token.encode('utf-8')
            # translate all bytes to their unicode string representation and flatten
            token_translated = ''.join(self.byte_encoder[b] for b in token_bytes)
            # perform all the applicable bpe merges according to self.bpe_ranks
            token_merged = self.bpe(token_translated).split(' ')
            # translate all bpe tokens to integers
            token_ix = [self.encoder[bpe_token] for bpe_token in token_merged]
            # extend our running list of all output integers
            bpe_idx.extend(token_ix)
        return bpe_idx

    def encode_and_show_work(self, text):
        """ debugging function, same as encode but returns all intermediate work """
        bpe_idx = []
        parts = []
        tokens = re.findall(self.pat, text)
        for token in tokens:
            token_bytes = token.encode('utf-8')
            token_translated = ''.join(self.byte_encoder[b] for b in token_bytes)
            token_merged = self.bpe(token_translated).split(' ')
            token_ix = [self.encoder[bpe_token] for bpe_token in token_merged]
            bpe_idx.extend(token_ix)
            parts.append({
                'token': token,
                'token_bytes': token_bytes,
                'token_translated': token_translated,
                'token_merged': token_merged,
                'token_ix': token_ix,
            })
        out = {
            'bpe_idx': bpe_idx, # the actual output sequence
            'tokens': tokens, # result of pre-tokenization
            'parts': parts, # intermediates for each token part
        }
        return out

    def decode(self, bpe_idx):
        """ list of integers comes in, string comes out """
        # inverse map the integers to get the tokens
        tokens_merged = [self.decoder[token] for token in bpe_idx]
        # inverse the byte encoder, e.g. recovering 'Ġ' -> ' ', and get the bytes
        tokens_flat = ''.join(tokens_merged)
        tokens_bytes = bytearray([self.byte_decoder[c] for c in tokens_flat])
        # recover the full utf-8 string
        text = tokens_bytes.decode('utf-8', errors='replace')
        return text

def get_file(local_file, remote_file):
    """ downloads remote_file to local_file if necessary """
    if not os.path.isfile(local_file):
        print(f"downloading {remote_file} to {local_file}")
        response = requests.get(remote_file)
        open(local_file, "wb").write(response.content)

def get_encoder():
    """
    Returns an instance of the GPT BPE Encoder/Decoder
    and handles caching of "database" files.
    """
    home_dir = os.path.expanduser('~')
    cache_dir = os.path.join(home_dir, '.cache', 'mingpt')
    os.makedirs(cache_dir, exist_ok=True)

    # load encoder.json that has the raw mappings from token -> bpe index
    encoder_local_file = os.path.join(cache_dir, 'encoder.json')
    encoder_remote_file = 'https://openaipublic.blob.core.windows.net/gpt-2/models/124M/encoder.json'
    get_file(encoder_local_file, encoder_remote_file)
    with open(encoder_local_file, 'r') as f:
        encoder = json.load(f)
    assert len(encoder) == 50257 # 256 individual byte tokens, 50,000 merged tokens, and 1 special <|endoftext|> token

    # load vocab.bpe that contains the bpe merges, i.e. the bpe tree structure
    # in the form tuples (a, b), that indicate that (a, b) is to be merged to one token ab
    vocab_local_file = os.path.join(cache_dir, 'vocab.bpe')
    vocab_remote_file = 'https://openaipublic.blob.core.windows.net/gpt-2/models/124M/vocab.bpe'
    get_file(vocab_local_file, vocab_remote_file)
    with open(vocab_local_file, 'r', encoding="utf-8") as f:
        bpe_data = f.read()
    # light postprocessing: strip the version on first line and the last line is a blank
    bpe_merges = [tuple(merge_str.split()) for merge_str in bpe_data.split('\n')[1:-1]]
    assert len(bpe_merges) == 50000 # 50,000 merged tokens

    # construct the Encoder object and return
    enc = Encoder(encoder, bpe_merges)
    return enc

# -----------------------------------------------------------------------------

class BPETokenizer:
    """ PyTorch-aware class that wraps the Encoder above """

    def __init__(self):
        self.encoder = get_encoder()

    @property
    def vocab_size(self):
        """Return the vocabulary size of the BPE tokenizer"""
        return len(self.encoder.encoder)  # 50257 tokens

    def encode(self, text):
        """Encode text to list of token IDs"""
        return self.encoder.encode(text)

    def __call__(self, text, return_tensors='pt'):
        # PyTorch only; here because we want to match huggingface/transformers interface
        assert return_tensors == 'pt'
        # single string input for now, in the future potentially a list of strings
        assert isinstance(text, str)
        # encode and create a "batch dimension" of 1
        idx = [self.encoder.encode(text)]
        # wrap into PyTorch tensor
        out = torch.tensor(idx, dtype=torch.long)
        return out

    def decode(self, idx):
        # ensure a simple 1D tensor for now
        assert idx.ndim == 1
        # decode indices to text
        text = self.encoder.decode(idx.tolist())
        return text

# Model Definitions (heavily edited based on karpathy/minGPT)
This section defines the neural network architectures used in the comparison. It includes the implementation of LSTM models with attention mechanisms and Transformer models. The block covers the setup of layers, activation functions, and any custom components required for the models.

References:
1) the official GPT-2 TensorFlow implementation released by OpenAI:
https://github.com/openai/gpt-2/blob/master/src/model.py
2) huggingface/transformers PyTorch implementation:
https://github.com/huggingface/transformers/blob/main/src/transformers/models/gpt2/modeling_gpt2.py

## Helper functions/submodule defintions

Here we define some useful functions/submodules for the main definitions below, especially the unified multi-head attention blocks which are used for both LSTM with Attention and the Transformer.

In [ ]:
import os
import sys
import json
import random
from ast import literal_eval

import numpy as np
import torch

# -----------------------------------------------------------------------------

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def setup_logging(config):
    """ monotonous bookkeeping """
    work_dir = config.system.work_dir
    # create the work directory if it doesn't already exist
    os.makedirs(work_dir, exist_ok=True)
    # log the args (if any)
    with open(os.path.join(work_dir, 'args.txt'), 'w') as f:
        f.write(' '.join(sys.argv))
    # log the config itself
    with open(os.path.join(work_dir, 'config.json'), 'w') as f:
        f.write(json.dumps(config.to_dict(), indent=4))

class CfgNode:
    """ a lightweight configuration class inspired by yacs """
    # TODO: convert to subclass from a dict like in yacs?
    # TODO: implement freezing to prevent shooting of own foot
    # TODO: additional existence/override checks when reading/writing params?

    def __init__(self, **kwargs):
        self.__dict__.update(kwargs)

    def __str__(self):
        return self._str_helper(0)

    def _str_helper(self, indent):
        """ need to have a helper to support nested indentation for pretty printing """
        parts = []
        for k, v in self.__dict__.items():
            if isinstance(v, CfgNode):
                parts.append("%s:\n" % k)
                parts.append(v._str_helper(indent + 1))
            else:
                parts.append("%s: %s\n" % (k, v))
        parts = [' ' * (indent * 4) + p for p in parts]
        return "".join(parts)

    def to_dict(self):
        """ return a dict representation of the config """
        return { k: v.to_dict() if isinstance(v, CfgNode) else v for k, v in self.__dict__.items() }

    def merge_from_dict(self, d):
        self.__dict__.update(d)

    def merge_from_args(self, args):
        """
        update the configuration from a list of strings that is expected
        to come from the command line, i.e. sys.argv[1:].

        The arguments are expected to be in the form of `--arg=value`, and
        the arg can use . to denote nested sub-attributes. Example:

        --model.n_layer=10 --trainer.batch_size=32
        """
        for arg in args:

            keyval = arg.split('=')
            assert len(keyval) == 2, "expecting each override arg to be of form --arg=value, got %s" % arg
            key, val = keyval # unpack

            # first translate val into a python object
            try:
                val = literal_eval(val)
                """
                need some explanation here.
                - if val is simply a string, literal_eval will throw a ValueError
                - if val represents a thing (like an 3, 3.14, [1,2,3], False, None, etc.) it will get created
                """
            except ValueError:
                pass

            # find the appropriate object to insert the attribute into
            assert key[:2] == '--'
            key = key[2:] # strip the '--'
            keys = key.split('.')
            obj = self
            for k in keys[:-1]:
                obj = getattr(obj, k)
            leaf_key = keys[-1]

            # ensure that this attribute exists
            assert hasattr(obj, leaf_key), f"{key} is not an attribute that exists in the config"

            # overwrite the attribute
            print("command line overwriting config attribute %s with %s" % (key, val))
            setattr(obj, leaf_key, val)

In [ ]:
import math

import torch
import torch.nn as nn
from torch.nn import functional as F

# -----------------------------------------------------------------------------

class NewGELU(nn.Module):
    """
    Implementation of the GELU activation function currently in Google BERT repo (identical to OpenAI GPT).
    Reference: Gaussian Error Linear Units (GELU) paper: https://arxiv.org/abs/1606.08415
    """
    def forward(self, x):
        return 0.5 * x * (1.0 + torch.tanh(math.sqrt(2.0 / math.pi) * (x + 0.044715 * torch.pow(x, 3.0))))

class UnifiedMultiHeadAttention(nn.Module):
    """
    Unified multi-head self-attention mechanism used by both LSTM and Transformer models.
    Supports both causal and bidirectional attention patterns.
    """
    def __init__(self, config, causal=False):
        super().__init__()
        assert config.n_embd % config.n_head == 0

        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.head_dim = config.n_embd // config.n_head
        self.causal = causal

        # Unified Q, K, V projections
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        # Output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)

        # Dropout - use different dropout rates based on config availability
        attn_dropout_rate = getattr(config, 'attn_pdrop', getattr(config, 'dropout', 0.1))
        resid_dropout_rate = getattr(config, 'resid_pdrop', getattr(config, 'dropout', 0.1))

        self.attn_dropout = nn.Dropout(attn_dropout_rate)
        self.resid_dropout = nn.Dropout(resid_dropout_rate)

        # Register causal mask buffer if needed
        if causal:
            # This will be filled in during forward pass based on sequence length
            self.register_buffer("causal_mask", None, persistent=False)

    def forward(self, x, attention_mask=None):
        B, T, C = x.size()  # batch size, sequence length, embedding dimensionality

        # Calculate Q, K, V for all heads in batch
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)

        # Reshape for multi-head attention: (B, T, n_head, head_dim) -> (B, n_head, T, head_dim)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        # Scaled dot-product attention
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(self.head_dim))

        # Apply causal mask if needed (for autoregressive models)
        if self.causal:
            if self.causal_mask is None or self.causal_mask.size(0) < T:
                # Create causal mask
                causal_mask = torch.tril(torch.ones(T, T, device=x.device, dtype=torch.bool))
                self.register_buffer("causal_mask", causal_mask, persistent=False)
            att = att.masked_fill(~self.causal_mask[:T, :T], float('-inf'))

        # Apply padding mask if provided
        if attention_mask is not None:
            # Convert attention mask to shape (B, 1, 1, T) for broadcasting
            mask = attention_mask.unsqueeze(1).unsqueeze(2)
            att = att.masked_fill(mask == 0, float('-inf'))

        # Softmax and dropout
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)

        # Apply attention to values
        y = att @ v  # (B, n_head, T, T) x (B, n_head, T, head_dim) -> (B, n_head, T, head_dim)

        # Concatenate heads: (B, n_head, T, head_dim) -> (B, T, n_head * head_dim)
        y = y.transpose(1, 2).contiguous().view(B, T, C)

        # Output projection and dropout
        y = self.resid_dropout(self.c_proj(y))
        return y

## LSTM w/ Attention Definition

In [ ]:
class LSTMAttentionBlock(nn.Module):
    """
    A single LSTM layer with self-attention mechanism using the unified attention implementation.
    Each block consists of:
    1. LSTM layer
    2. Unified multi-head self-attention
    3. Residual connection and layer normalization
    """
    def __init__(self, config):
        super().__init__()
        self.hidden_size = config.n_embd

        # LSTM layer
        self.lstm = nn.LSTM(self.hidden_size, self.hidden_size, 1,
                           batch_first=True, dropout=0)

        # Unified multi-head self-attention (same as used in transformer)
        self.attn = UnifiedMultiHeadAttention(config, causal=False)

        # Layer normalization
        self.ln1 = nn.LayerNorm(self.hidden_size)
        self.ln2 = nn.LayerNorm(self.hidden_size)

    def forward(self, x, attention_mask=None):
        # Store residual for later
        residual = x

        # 1. LSTM processing with residual connection
        lstm_out, _ = self.lstm(x)
        x = self.ln1(lstm_out + residual)

        # 2. Multi-head self-attention with residual connection
        residual = x
        attn_out = self.attn(x, attention_mask)
        x = self.ln2(attn_out + residual)

        return x

class LSTMClassifier(nn.Module):
    """ LSTM-based text classifier with per-layer attention mechanism (similar to transformer architecture)"""
    @staticmethod
    def get_default_config():
        C = CfgNode()
        # either model_type or (n_layer, n_embd) must be given in the config
        C.model_type = 'lstm'
        C.n_layer = None
        C.n_embd = None
        C.n_head = 4  # Number of attention heads
        # these options must be filled in externally
        C.vocab_size = None
        C.block_size = None
        # dropout hyperparameters (compatible with GPT config names)
        C.dropout = 0.3
        C.embd_pdrop = 0.1  # Embedding dropout (same as GPT)
        C.resid_pdrop = 0.1  # Residual dropout (same as GPT)
        C.attn_pdrop = 0.1   # Attention dropout (same as GPT)
        # classification specific
        C.num_classes = 2  # for binary sentiment classification
        return C

    def __init__(self, config):
        super().__init__()
        # basic sanity checks
        assert config.vocab_size is not None, "config.vocab_size must be set for LSTMClassifier"
        assert config.block_size is not None, "config.block_size must be set for LSTMClassifier"
        assert config.n_layer is not None, "config.n_layer must be set for LSTMClassifier"
        assert config.n_embd is not None, "config.n_embd must be set for LSTMClassifier"

        self.config = config

        # Embedding layer with dropout (matching GPT architecture)
        self.embedding = nn.Embedding(config.vocab_size, config.n_embd)
        self.embd_dropout = nn.Dropout(getattr(config, 'embd_pdrop', config.dropout))

        # Stack of LSTM-Attention blocks
        self.layers = nn.ModuleList([
            LSTMAttentionBlock(config) for _ in range(config.n_layer)
        ])

        # Final layer norm
        self.ln_f = nn.LayerNorm(config.n_embd)

        # Dropout and classification head
        self.dropout = nn.Dropout(config.dropout)
        self.classifier = nn.Linear(config.n_embd, config.num_classes)

        # Language modeling head for pre-training (shares weights with embedding)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        # Tie weights between word embeddings and language modeling head (same as GPT)
        self.lm_head.weight = self.embedding.weight

        # Initialize weights
        self.apply(self._init_weights)

        # report number of parameters
        n_params = sum(p.numel() for p in self.parameters())
        print("LSTM with Attention: number of parameters: %.2fM" % (n_params/1e6,))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.LayerNorm):
            torch.nn.init.zeros_(module.bias)
            torch.nn.init.ones_(module.weight)

    def forward(self, input_ids, attention_mask, labels=None):
        # Embed tokens with dropout (matching GPT architecture)
        x = self.embedding(input_ids)  # (batch, seq_len, n_embd)
        x = self.embd_dropout(x)

        # Ensure attention_mask is float on the same device/dtype
        if attention_mask is not None:
            attention_mask = attention_mask.to(dtype=x.dtype, device=x.device)
            # Apply attention mask to embeddings (zero out padded tokens)
            x = x * attention_mask.unsqueeze(-1)

        # Pass through each LSTM-Attention block
        for layer in self.layers:
            x = layer(x, attention_mask)

        # Final layer normalization
        x = self.ln_f(x)

        # Global average pooling with attention mask
        if attention_mask is not None:
            # Apply attention mask and compute mean over valid tokens
            mask_expanded = attention_mask.unsqueeze(-1).expand_as(x)
            x_masked = x * mask_expanded
            denom = attention_mask.sum(dim=1, keepdim=True).to(x.dtype) + 1e-8
            pooled = x_masked.sum(dim=1) / denom
        else:
            # Simple mean pooling if no attention mask
            pooled = x.mean(dim=1)

        # Classification
        pooled = self.dropout(pooled)
        logits = self.classifier(pooled)

        # Calculate loss if labels are provided
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels)

        return logits, loss

    def get_hidden_states(self, input_ids, attention_mask=None):
        """Get hidden states for language modeling"""
        # Embed tokens with dropout
        x = self.embedding(input_ids)
        x = self.embd_dropout(x)

        # Ensure attention_mask is float on the same device/dtype
        if attention_mask is not None:
            attention_mask = attention_mask.to(dtype=x.dtype, device=x.device)
            # Apply attention mask to embeddings (zero out padded tokens)
            x = x * attention_mask.unsqueeze(-1)

        # Pass through each LSTM-Attention block
        for layer in self.layers:
            x = layer(x, attention_mask)

        # Final layer normalization
        x = self.ln_f(x)

        return x

    def forward_lm(self, input_ids, attention_mask=None):
        """Forward pass for language modeling (pre-training)"""
        hidden_states = self.get_hidden_states(input_ids, attention_mask)
        logits = self.lm_head(hidden_states)
        return logits

## Transformer Definition

In [ ]:
class BidirectionalSelfAttention(UnifiedMultiHeadAttention):
    """
    Bidirectional self-attention layer using the unified attention mechanism.
    This is a wrapper for backward compatibility with existing transformer code.
    """
    def __init__(self, config):
        super().__init__(config, causal=False)  # Bidirectional = non-causal

class Block(nn.Module):
    """ an unassuming Transformer block """

    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = BidirectionalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = nn.ModuleDict(dict(
            c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd),
            c_proj  = nn.Linear(4 * config.n_embd, config.n_embd),
            act     = NewGELU(),
            dropout = nn.Dropout(config.resid_pdrop),
        ))
        m = self.mlp
        # ModuleDict does not expose keys as attributes; use indexing
        self.mlpf = lambda x: m['dropout'](m['c_proj'](m['act'](m['c_fc'](x)))) # MLP forward

    def forward(self, x, attention_mask=None):
        x = x + self.attn(self.ln_1(x), attention_mask)
        x = x + self.mlpf(self.ln_2(x))
        return x


class GPT(nn.Module):
    """ Transformer modified for sentiment classification """

    @staticmethod
    def get_default_config():
        C = CfgNode()
        # either model_type or (n_layer, n_head, n_embd) must be given in the config
        C.model_type = 'gpt'
        C.n_layer = None
        C.n_head = None
        C.n_embd =  None
        # these options must be filled in externally
        C.vocab_size = None
        C.block_size = None
        # dropout hyperparameters
        C.embd_pdrop = 0.1
        C.resid_pdrop = 0.1
        C.attn_pdrop = 0.1
        # classification specific
        C.num_classes = 2  # for binary sentiment classification
        return C

    def __init__(self, config):
        super().__init__()
        assert config.vocab_size is not None
        assert config.block_size is not None
        self.block_size = config.block_size

        type_given = config.model_type is not None
        params_given = all([config.n_layer is not None, config.n_head is not None, config.n_embd is not None])
        assert type_given ^ params_given # exactly one of these (XOR)
        if type_given:
            # translate from model_type to detailed configuration
            config.merge_from_dict({
                # names follow the huggingface naming conventions
                # GPT-1
                'openai-gpt':   dict(n_layer=12, n_head=12, n_embd=768),  # 117M params
                # GPT-2 configs
                'gpt2':         dict(n_layer=12, n_head=12, n_embd=768),  # 124M params
                'gpt2-medium':  dict(n_layer=24, n_head=16, n_embd=1024), # 350M params
                'gpt2-large':   dict(n_layer=36, n_head=20, n_embd=1280), # 774M params
                'gpt2-xl':      dict(n_layer=48, n_head=25, n_embd=1600), # 1558M params
                # Gophers
                'gopher-44m':   dict(n_layer=8, n_head=16, n_embd=512),
                # (there are a number more...)
                # I made these tiny models up
                'gpt-mini':     dict(n_layer=6, n_head=6, n_embd=192),
                'gpt-micro':    dict(n_layer=4, n_head=4, n_embd=128),
                'gpt-nano':     dict(n_layer=3, n_head=3, n_embd=48),
            }[config.model_type])

        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            drop = nn.Dropout(config.embd_pdrop),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),
        ))

        # Classification head for fine-tuning
        self.classifier = nn.Linear(config.n_embd, config.num_classes)
        self.dropout = nn.Dropout(config.embd_pdrop)

        # Language modeling head for pre-training
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        # Tie weights between word embeddings and language modeling head
        self.lm_head.weight = self.transformer.wte.weight

        # init all weights, and apply a special scaled init to the residual projections, per GPT-2 paper
        self.apply(self._init_weights)
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02/math.sqrt(2 * config.n_layer))

        # report number of parameters
        n_params = sum(p.numel() for p in self.parameters())
        print("Transformer: number of parameters: %.2fM" % (n_params/1e6,))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.LayerNorm):
            torch.nn.init.zeros_(module.bias)
            torch.nn.init.ones_(module.weight)

    @classmethod
    def from_pretrained(cls, model_type):
        """
        Initialize a pretrained GPT model by copying over the weights
        from a huggingface/transformers checkpoint.
        """
        assert model_type in {'gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl'}
        from transformers import GPT2LMHeadModel

        # create a from-scratch initialized minGPT model
        config = cls.get_default_config()
        config.model_type = model_type
        config.vocab_size = 50257 # openai's model vocabulary
        config.block_size = 1024  # openai's model block_size
        model = GPT(config)
        sd = model.state_dict()

        # init a huggingface/transformers model
        model_hf = GPT2LMHeadModel.from_pretrained(model_type)
        sd_hf = model_hf.state_dict()

        # copy while ensuring all of the parameters are aligned and match in names and shapes
        keys = [k for k in sd_hf if not k.endswith('attn.masked_bias')] # ignore these
        transposed = ['attn.c_attn.weight', 'attn.c_proj.weight', 'mlp.c_fc.weight', 'mlp.c_proj.weight']
        # basically the openai checkpoints use a "Conv1D" module, but we only want to use a vanilla nn.Linear.
        # this means that we have to transpose these weights when we import them
        assert len(keys) == len(sd)
        for k in keys:
            if any(k.endswith(w) for w in transposed):
                # special treatment for the Conv1D weights we need to transpose
                assert sd_hf[k].shape[::-1] == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k].t())
            else:
                # vanilla copy over the other parameters
                assert sd_hf[k].shape == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k])

        return model

    def configure_optimizers(self, train_config):
        """
        This long function is unfortunately doing something very simple and is being very defensive:
        We are separating out all parameters of the model into two buckets: those that will experience
        weight decay for regularization and those that won't (biases, and layernorm/embedding weights).
        We are then returning the PyTorch optimizer object.
        """

        # separate out all parameters to those that will and won't experience regularizing weight decay
        decay = set()
        no_decay = set()
        whitelist_weight_modules = (torch.nn.Linear, )
        blacklist_weight_modules = (torch.nn.LayerNorm, torch.nn.Embedding)
        for mn, m in self.named_modules():
            for pn, p in m.named_parameters():
                fpn = '%s.%s' % (mn, pn) if mn else pn # full param name
                # random note: because named_modules and named_parameters are recursive
                # we will see the same tensors p many many times. but doing it this way
                # allows us to know which parent module any tensor p belongs to...
                if pn.endswith('bias'):
                    # all biases will not be decayed
                    no_decay.add(fpn)
                elif pn.endswith('weight') and isinstance(m, whitelist_weight_modules):
                    # weights of whitelist modules will be weight decayed
                    decay.add(fpn)
                elif pn.endswith('weight') and isinstance(m, blacklist_weight_modules):
                    # weights of blacklist modules will NOT be weight decayed
                    no_decay.add(fpn)

        # validate that we considered every parameter
        param_dict = {pn: p for pn, p in self.named_parameters()}
        inter_params = decay & no_decay
        union_params = decay | no_decay
        assert len(inter_params) == 0, "parameters %s made it into both decay/no_decay sets!" % (str(inter_params), )
        assert len(param_dict.keys() - union_params) == 0, "parameters %s were not separated into either decay/no_decay set!" \
                                                    % (str(param_dict.keys() - union_params), )

        # create the pytorch optimizer object
        optim_groups = [
            {"params": [param_dict[pn] for pn in sorted(list(decay))], "weight_decay": train_config.weight_decay},
            {"params": [param_dict[pn] for pn in sorted(list(no_decay))], "weight_decay": 0.0},
        ]
        optimizer = torch.optim.AdamW(optim_groups, lr=train_config.learning_rate, betas=train_config.betas)
        return optimizer

    def forward(self, input_ids, attention_mask=None, labels=None):
        device = input_ids.device
        b, t = input_ids.size()
        assert t <= self.block_size, f"Cannot forward sequence of length {t}, block size is only {self.block_size}"
        pos = torch.arange(0, t, dtype=torch.long, device=device).unsqueeze(0) # shape (1, t)

        # forward the GPT model itself
        tok_emb = self.transformer.wte(input_ids) # token embeddings of shape (b, t, n_embd)
        pos_emb = self.transformer.wpe(pos) # position embeddings of shape (1, t, n_embd)
        x = self.transformer.drop(tok_emb + pos_emb)

        # ensure attention_mask is float on same device/dtype when used for pooling or passed to blocks
        if attention_mask is not None:
            attention_mask = attention_mask.to(dtype=x.dtype, device=x.device)
        for block in self.transformer.h:
            x = block(x, attention_mask)
        x = self.transformer.ln_f(x)

        # Global average pooling with attention mask
        if attention_mask is not None:
            # Apply attention mask and compute mean over valid tokens
            mask_expanded = attention_mask.unsqueeze(-1).expand_as(x)
            x_masked = x * mask_expanded
            denom = attention_mask.sum(dim=1, keepdim=True).to(x.dtype) + 1e-8
            pooled = x_masked.sum(dim=1) / denom
        else:
            # Simple mean pooling if no attention mask
            pooled = x.mean(dim=1)

        # Classification
        pooled = self.dropout(pooled)
        logits = self.classifier(pooled)

        # Calculate loss if labels are provided
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels)

        return logits, loss

    def get_hidden_states(self, input_ids):
        """Get hidden states for language modeling (used during pre-training)."""
        device = input_ids.device
        b, t = input_ids.size()
        assert t <= self.block_size, f"Cannot forward sequence of length {t}, block size is only {self.block_size}"
        pos = torch.arange(0, t, dtype=torch.long, device=device).unsqueeze(0)

        # Forward through transformer
        tok_emb = self.transformer.wte(input_ids)
        pos_emb = self.transformer.wpe(pos)
        x = self.transformer.drop(tok_emb + pos_emb)

        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)

        return x

    def forward_lm(self, input_ids, attention_mask=None):
        """Forward pass for language modeling (next token prediction)."""
        hidden_states = self.get_hidden_states(input_ids)
        return self.lm_head(hidden_states)


# Training
This section contains the logic for training the models. It includes setting up the training loop, specifying loss functions, optimizers, and evaluation metrics. The section will also handle logging, checkpointing, and validation during training to monitor model performance.

## Configuration
Here we set the basic (hyper-)parameters for training.

In [ ]:
"""
Comprehensive training script for LSTM and GPT models with pre-training and fine-tuning.
"""

import json
import time
from pathlib import Path
from typing import Dict, List

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torch.optim import Adam
import numpy as np
from tqdm import tqdm
from datasets import load_dataset
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score
import seaborn as sns


# Global configuration
class Config:
    # Model hyperparameters
    BATCH_SIZE = 4
    PRETRAIN_EPOCHS = 8
    FINETUNE_EPOCHS = 3
    PRETRAIN_LR = 5e-4
    FINETUNE_LR = 1e-4
    MAX_LEN = 512
    EMBEDDING_DIM = 32
    HIDDEN_DIM = 64
    NUM_LAYERS = 1
    NUM_HEADS = 1
    DROPOUT = 0.1

    # Data configuration
    WIKI_SUBSET_SIZE = 5  # Number of Wikipedia articles for pre-training

    # Paths
    CHECKPOINT_DIR = Path("checkpoints")
    LOGS_DIR = Path("logs")
    RESULTS_DIR = Path("results")

## Dataset initialization
Here we import the wikipedia dataset (https://huggingface.co/datasets/wikimedia/wikipedia) for pre-training and the IMDB dataset (https://huggingface.co/datasets/stanfordnlp/imdb) for fine-tuning for the sentiment task.

In [ ]:
class WikiTextDataset(Dataset):
    """Dataset for Wikipedia pre-training with next-token prediction"""

    def __init__(self, texts: List[str], tokenizer: BPETokenizer, max_len: int = 512):
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.data = []

        print(f"Processing {len(texts)} Wikipedia texts...")
        for text in tqdm(texts, desc="Tokenizing"):
            tokens = tokenizer.encode(text)

            # Create sliding windows for language modeling
            for i in range(0, len(tokens) - max_len, max_len // 2):
                sequence = tokens[i:i + max_len]
                if len(sequence) == max_len:
                    self.data.append(sequence)

        print(f"Created {len(self.data)} training sequences")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sequence = self.data[idx]
        # For language modeling: input is sequence[:-1], target is sequence[1:]
        input_ids = sequence[:-1]
        targets = sequence[1:]

        # Create attention mask (all 1s for valid tokens)
        attention_mask = [1] * len(input_ids)

        return {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
            'labels': torch.tensor(targets, dtype=torch.long)
        }

class IMDBDataset(Dataset):
    """Dataset for IMDB sentiment classification fine-tuning"""

    def __init__(self, dataset, tokenizer: BPETokenizer, max_len: int = 512):
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.data = []

        print(f"Processing {len(dataset)} IMDB examples...")
        for example in tqdm(dataset, desc="Tokenizing IMDB"):
            tokens = tokenizer.encode(example['text'])

            # Truncate or pad to max_len
            if len(tokens) > max_len:
                tokens = tokens[:max_len]

            attention_mask = [1] * len(tokens) + [0] * (max_len - len(tokens))
            tokens = tokens + [0] * (max_len - len(tokens))

            self.data.append({
                'input_ids': tokens,
                'attention_mask': attention_mask,
                'label': example['label']
            })

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return {
            'input_ids': torch.tensor(self.data[idx]['input_ids'], dtype=torch.long),
            'attention_mask': torch.tensor(self.data[idx]['attention_mask'], dtype=torch.long),
            'label': torch.tensor(self.data[idx]['label'], dtype=torch.long)
        }


## Training class

Here we define a Trainer class to facilitate pre-training and fine-tuning, take cares of evaluation and loading previous checkpoints.

In [ ]:
class Trainer:
    """Unified trainer for both pre-training and fine-tuning"""

    def __init__(self, model, tokenizer, device, config: Config):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device
        self.config = config

        # Create directories
        self.config.CHECKPOINT_DIR.mkdir(exist_ok=True)
        self.config.LOGS_DIR.mkdir(exist_ok=True)
        self.config.RESULTS_DIR.mkdir(exist_ok=True)

        # Initialize evaluation tracking
        self.evaluation_history = {
            'pretrain': [],  # Pre-training evaluation data
            'finetune': []   # Fine-tuning evaluation data
        }
        self.model_name = type(self.model).__name__.lower()

        # Get model parameter count and display name
        self.total_parameters = self.count_parameters()
        self.display_name = self.get_model_display_name()

        # Initialize timing tracking
        self.training_start_time = None
        self.epoch_times = []

        print(f"Initialized {self.display_name} with {self.total_parameters:,} parameters")

    def count_parameters(self) -> int:
        """Count total number of parameters in the model"""
        return sum(p.numel() for p in self.model.parameters())

    def get_model_display_name(self) -> str:
        """Get the display name for the model (LSTM -> LSTM with Attention, GPT -> Transformer)"""
        if 'lstm' in self.model_name:
            return 'LSTM with Attention'
        elif 'gpt' in self.model_name:
            return 'Transformer'
        else:
            return self.model_name.upper()

    def pretrain(self, wiki_texts: List[str], epochs: int, lr: float) -> Dict[str, List[float]]:
        """Pre-train model on Wikipedia text using language modeling objective"""
        print(f"\n{'='*60}")
        print(f"PRE-TRAINING {self.display_name} MODEL")
        print(f"{'='*60}")

        # Create dataset and dataloader
        dataset = WikiTextDataset(wiki_texts, self.tokenizer, self.config.MAX_LEN - 1)
        dataloader = DataLoader(dataset, batch_size=self.config.BATCH_SIZE, shuffle=True)

        # Setup optimizer and loss function
        optimizer = Adam(self.model.parameters(), lr=lr)
        criterion = nn.CrossEntropyLoss(ignore_index=0)  # Ignore padding tokens

        # Training metrics
        metrics = {'train_loss': [], 'perplexity': [], 'epoch_times': []}

        # Start timing
        self.training_start_time = time.time()

        self.model.train()
        for epoch in range(epochs):
            epoch_start_time = time.time()
            epoch_loss = 0
            epoch_tokens = 0

            with tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}") as pbar:
                for batch in pbar:
                    input_ids = batch['input_ids'].to(self.device)
                    attention_mask = batch['attention_mask'].to(self.device)
                    targets = batch['labels'].to(self.device)

                    optimizer.zero_grad()

                    # Forward pass - get logits for next token prediction
                    if hasattr(self.model, 'forward_lm'):  # Language modeling forward
                        logits = self.model.forward_lm(input_ids, attention_mask)
                    else:
                        # Fallback: use regular forward and extract hidden states
                        hidden_states = self.model.get_hidden_states(input_ids, attention_mask)
                        # Project to vocabulary for next token prediction
                        logits = self.model.lm_head(hidden_states)

                    # Calculate loss
                    loss = criterion(logits.view(-1, logits.size(-1)), targets.view(-1))

                    loss.backward()
                    optimizer.step()

                    # Update metrics
                    epoch_loss += loss.item()
                    epoch_tokens += attention_mask.sum().item()

                    # Update progress bar
                    pbar.set_postfix({
                        'loss': f"{loss.item():.4f}",
                        'ppl': f"{torch.exp(loss):.2f}"
                    })

            # Calculate epoch metrics and timing
            epoch_time = time.time() - epoch_start_time
            elapsed_time = time.time() - self.training_start_time

            avg_loss = epoch_loss / len(dataloader)
            perplexity = np.exp(avg_loss)

            metrics['train_loss'].append(avg_loss)
            metrics['perplexity'].append(perplexity)
            metrics['epoch_times'].append(elapsed_time)

            print(f"Epoch {epoch+1}: Loss={avg_loss:.4f}, Perplexity={perplexity:.2f}, Time={epoch_time:.1f}s (Total: {elapsed_time:.1f}s)")

            # Save evaluation data
            epoch_metrics = {
                'train_loss': avg_loss,
                'perplexity': perplexity,
                'total_tokens': epoch_tokens,
                'epoch_time': epoch_time,
                'elapsed_time': elapsed_time
            }
            additional_data = {
                'learning_rate': lr,
                'batch_size': self.config.BATCH_SIZE,
                'dataset_size': len(dataloader.dataset),
                'total_parameters': self.total_parameters,
                'model_display_name': self.display_name
            }
            self.save_evaluation_data('pretrain', epoch+1, epoch_metrics, additional_data)

            # Save checkpoint
            self.save_checkpoint(f"{type(self.model).__name__.lower()}_pretrained_epoch_{epoch+1}.pt", epoch+1)

        # Save final pre-training summary
        summary = self.create_evaluation_summary()
        summary_file = self.config.RESULTS_DIR / f"{self.model_name}_pretrain_summary.json"
        with open(summary_file, 'w') as f:
            json.dump(summary, f, indent=2)
        print(f"Pre-training summary saved: {summary_file}")

        return metrics

    def finetune(self, train_loader, val_loader, epochs: int, lr: float) -> Dict[str, List[float]]:
        """Fine-tune model on IMDB sentiment classification"""
        print(f"\n{'='*60}")
        print(f"FINE-TUNING {self.display_name} MODEL")
        print(f"{'='*60}")

        optimizer = Adam(self.model.parameters(), lr=lr)

        metrics = {
            'train_loss': [], 'train_acc': [],
            'val_loss': [], 'val_acc': [], 'val_f1': [],
            'epoch_times': []
        }

        # Start timing
        self.training_start_time = time.time()

        best_val_acc = 0

        for epoch in range(epochs):
            epoch_start_time = time.time()

            # Training
            self.model.train()
            train_loss = 0
            train_correct = 0
            train_total = 0

            with tqdm(train_loader, desc=f"Training Epoch {epoch+1}/{epochs}") as pbar:
                for batch in pbar:
                    input_ids = batch['input_ids'].to(self.device)
                    attention_mask = batch['attention_mask'].to(self.device)
                    labels = batch['label'].to(self.device)

                    optimizer.zero_grad()
                    logits, loss = self.model(input_ids, attention_mask, labels)
                    loss.backward()
                    optimizer.step()

                    train_loss += loss.item()
                    predictions = torch.argmax(logits, dim=1)
                    train_correct += (predictions == labels).sum().item()
                    train_total += labels.size(0)

                    pbar.set_postfix({
                        'loss': f"{loss.item():.4f}",
                        'acc': f"{train_correct/train_total:.4f}"
                    })

            # Validation
            val_metrics = self.evaluate(val_loader)

            # Calculate timing
            epoch_time = time.time() - epoch_start_time
            elapsed_time = time.time() - self.training_start_time

            # Store metrics
            train_loss_avg = train_loss / len(train_loader)
            train_acc_avg = train_correct / train_total

            metrics['train_loss'].append(train_loss_avg)
            metrics['train_acc'].append(train_acc_avg)
            metrics['val_loss'].append(val_metrics['loss'])
            metrics['val_acc'].append(val_metrics['accuracy'])
            metrics['val_f1'].append(val_metrics['f1'])
            metrics['epoch_times'].append(elapsed_time)

            print(f"Epoch {epoch+1}:")
            print(f"  Train Loss: {train_loss_avg:.4f}, Train Acc: {train_acc_avg:.4f}")
            print(f"  Val Loss: {val_metrics['loss']:.4f}, Val Acc: {val_metrics['accuracy']:.4f}, Val F1: {val_metrics['f1']:.4f}")
            print(f"  Time: {epoch_time:.1f}s (Total: {elapsed_time:.1f}s)")

            # Save evaluation data
            epoch_metrics = {
                'train_loss': train_loss_avg,
                'train_acc': train_acc_avg,
                'val_loss': val_metrics['loss'],
                'val_acc': val_metrics['accuracy'],
                'val_f1': val_metrics['f1'],
                'val_precision': val_metrics['precision'],
                'val_recall': val_metrics['recall'],
                'epoch_time': epoch_time,
                'elapsed_time': elapsed_time
            }
            additional_data = {
                'learning_rate': lr,
                'batch_size': self.config.BATCH_SIZE,
                'train_dataset_size': len(train_loader.dataset),
                'val_dataset_size': len(val_loader.dataset),
                'is_best_epoch': val_metrics['accuracy'] > best_val_acc,
                'total_parameters': self.total_parameters,
                'model_display_name': self.display_name
            }
            self.save_evaluation_data('finetune', epoch+1, epoch_metrics, additional_data)

            # Save best model
            if val_metrics['accuracy'] > best_val_acc:
                best_val_acc = val_metrics['accuracy']
                self.save_checkpoint(f"{type(self.model).__name__.lower()}_finetuned_best.pt", epoch+1, is_best=True)

            # Save checkpoint for this epoch
                self.save_checkpoint(f"{type(self.model).__name__.lower()}_finetuned_best.pt", epoch+1, is_best=True)

            # Save checkpoint for this epoch
            self.save_checkpoint(f"{type(self.model).__name__.lower()}_finetuned_epoch_{epoch+1}.pt", epoch+1)

        # Save final training summary
        summary = self.create_evaluation_summary()
        summary_file = self.config.RESULTS_DIR / f"{self.model_name}_training_summary.json"
        with open(summary_file, 'w') as f:
            json.dump(summary, f, indent=2)
        print(f"Training summary saved: {summary_file}")

        return metrics

    def evaluate(self, dataloader) -> Dict[str, float]:
        """Evaluate model on given dataloader"""
        self.model.eval()
        total_loss = 0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for batch in tqdm(dataloader, desc="Evaluating", leave=False):
                input_ids = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels = batch['label'].to(self.device)

                logits, loss = self.model(input_ids, attention_mask, labels)

                total_loss += loss.item()
                predictions = torch.argmax(logits, dim=1)

                all_preds.extend(predictions.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        accuracy = np.mean(np.array(all_preds) == np.array(all_labels))
        f1 = f1_score(all_labels, all_preds, average='binary')
        precision = precision_score(all_labels, all_preds, average='binary')
        recall = recall_score(all_labels, all_preds, average='binary')

        return {
            'loss': total_loss / len(dataloader),
            'accuracy': accuracy,
            'f1': f1,
            'precision': precision,
            'recall': recall,
            'predictions': all_preds,
            'labels': all_labels
        }

    def save_checkpoint(self, filename: str, epoch: int, is_best: bool = False):
        """Save model checkpoint"""
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': self.model.state_dict(),
            'model_config': self.model.config if hasattr(self.model, 'config') else None,
            'model_type': type(self.model).__name__
        }

        filepath = self.config.CHECKPOINT_DIR / filename
        torch.save(checkpoint, filepath)

        if is_best:
            best_path = self.config.CHECKPOINT_DIR / f"{type(self.model).__name__.lower()}_best.pt"
            torch.save(checkpoint, best_path)

        print(f"Checkpoint saved: {filepath}")

    def save_evaluation_data(self, phase: str, epoch: int, metrics: Dict, additional_data: Dict = None):
        """Save evaluation data for current epoch"""
        # Convert metrics to ensure JSON serialization
        serializable_metrics = {}
        for key, value in metrics.items():
            if isinstance(value, (int, float, str, bool)):
                serializable_metrics[key] = float(value) if isinstance(value, (int, float)) else value
            else:
                serializable_metrics[key] = str(value)

        eval_data = {
            'epoch': epoch,
            'phase': phase,
            'model_name': self.model_name,
            'timestamp': time.time(),
            'metrics': serializable_metrics
        }

        # Add any additional data (e.g., learning rate, batch size, etc.)
        if additional_data:
            serializable_additional = {}
            for key, value in additional_data.items():
                if isinstance(value, (int, float, str, bool)):
                    serializable_additional[key] = value
                else:
                    serializable_additional[key] = str(value)
            eval_data.update(serializable_additional)

        # Store in history
        self.evaluation_history[phase].append(eval_data)

        # Save to JSON file immediately (incremental save)
        eval_file = self.config.RESULTS_DIR / f"{self.model_name}_{phase}_evaluation.json"
        with open(eval_file, 'w') as f:
            json.dump(self.evaluation_history[phase], f, indent=2)

        print(f"Evaluation data saved: {eval_file}")

    def create_evaluation_summary(self) -> Dict:
        """Create comprehensive evaluation summary"""
        summary = {
            'model_name': self.model_name,
            'total_pretrain_epochs': len(self.evaluation_history['pretrain']),
            'total_finetune_epochs': len(self.evaluation_history['finetune']),
            'pretrain_history': self.evaluation_history['pretrain'],
            'finetune_history': self.evaluation_history['finetune']
        }

        # Calculate best performances
        if self.evaluation_history['pretrain']:
            best_pretrain = min(self.evaluation_history['pretrain'],
                              key=lambda x: x['metrics']['train_loss'])
            summary['best_pretrain'] = {
                'epoch': best_pretrain['epoch'],
                'loss': best_pretrain['metrics']['train_loss'],
                'perplexity': best_pretrain['metrics']['perplexity']
            }

        if self.evaluation_history['finetune']:
            best_finetune = max(self.evaluation_history['finetune'],
                              key=lambda x: x['metrics']['val_f1'])
            summary['best_finetune'] = {
                'epoch': best_finetune['epoch'],
                'accuracy': best_finetune['metrics']['val_acc'],
                'f1_score': best_finetune['metrics']['val_f1'],
                'precision': best_finetune['metrics'].get('val_precision', 0),
                'recall': best_finetune['metrics'].get('val_recall', 0)
            }

        return summary

    def load_checkpoint(self, filepath: str, strict: bool = True) -> int:
        """Load model checkpoint"""
        checkpoint = torch.load(filepath, map_location=self.device, weights_only=False)

        if not strict:
            # Load only compatible layers (skip classifier/lm_head mismatches)
            model_dict = self.model.state_dict()
            pretrained_dict = checkpoint['model_state_dict']

            # Filter out incompatible keys
            compatible_dict = {}
            for k, v in pretrained_dict.items():
                if k in model_dict and model_dict[k].shape == v.shape:
                    compatible_dict[k] = v
                else:
                    print(f"Skipping incompatible layer: {k} (shapes: model={model_dict.get(k, 'missing').shape if k in model_dict else 'missing'}, checkpoint={v.shape})")

            # Update model dict and load
            model_dict.update(compatible_dict)
            self.model.load_state_dict(model_dict)
            print(f"Loaded {len(compatible_dict)} compatible layers from checkpoint")
        else:
            self.model.load_state_dict(checkpoint['model_state_dict'])

        epoch = checkpoint['epoch']
        print(f"Loaded checkpoint from epoch {epoch}: {filepath}")
        return epoch

## Helper functions

Here we define some helper functions to make the code for the main training loop more readable.

In [ ]:
def create_model(model_type: str, tokenizer: BPETokenizer, config: Config, for_pretraining: bool = False):
    """Create and configure model for either pre-training or fine-tuning"""
    vocab_size = tokenizer.vocab_size + 2

    if model_type.lower() == 'lstm':
        model_config = LSTMClassifier.get_default_config()
        model_config.vocab_size = vocab_size
        model_config.block_size = config.HIDDEN_DIM
        model_config.n_embd = config.EMBEDDING_DIM
        model_config.n_layer = config.NUM_LAYERS
        model_config.n_head = config.NUM_HEADS
        model_config.dropout = config.DROPOUT
        model_config.embd_pdrop = config.DROPOUT
        model_config.resid_pdrop = config.DROPOUT
        model_config.attn_pdrop = config.DROPOUT
        model_config.num_classes = 2  # Always 2 for classification head

        model = LSTMClassifier(model_config)

    elif model_type.lower() == 'gpt':
        model_config = GPT.get_default_config()
        model_config.model_type = None
        model_config.vocab_size = vocab_size
        model_config.block_size = config.MAX_LEN
        model_config.n_embd = config.EMBEDDING_DIM
        model_config.n_head = config.NUM_HEADS
        model_config.n_layer = config.NUM_LAYERS
        model_config.embd_pdrop = config.DROPOUT
        model_config.resid_pdrop = config.DROPOUT
        model_config.attn_pdrop = config.DROPOUT
        model_config.num_classes = 2  # Always 2 for classification head

        model = GPT(model_config)
    else:
        raise ValueError(f"Unknown model type: {model_type}")

    return model

def load_wikipedia_data(subset_size: int = 50000) -> List[str]:
    """Load Wikipedia dataset for pre-training"""
    print(f"Loading Wikipedia dataset (subset of {subset_size} articles)...")

    try:
        # Load Wikipedia dataset
        wiki_dataset = load_dataset("wikimedia/wikipedia", "20231101.en", split=f"train[:{subset_size}]")
        texts = [article['text'] for article in wiki_dataset if len(article['text']) > 100]
        print(f"Loaded {len(texts)} Wikipedia articles")
        return texts
    except Exception as e:
        print(f"Error loading Wikipedia dataset: {e}")
        print("Using fallback smaller dataset...")
        # Fallback to a smaller dataset
        try:
            wiki_dataset = load_dataset("wikitext", "wikitext-103-raw-v1", split="train")
            texts = [article['text'] for article in wiki_dataset if len(article['text']) > 100]
            return texts[:subset_size]
        except:
            # Ultimate fallback - create dummy data
            print("Using dummy data for testing...")
            return [f"This is a sample Wikipedia article number {i} with some content." * 10 for i in range(1000)]

def load_imdb_data():
    """Load IMDB dataset for fine-tuning"""
    print("Loading IMDB dataset...")
    dataset = load_dataset("imdb")
    return dataset["train"], dataset["test"]

def plot_metrics(metrics: Dict[str, List[float]], model_name: str, phase: str, total_params: int = None):
    """Plot training metrics including time-based plots"""
    # Convert model name if needed
    display_name = model_name
    if 'lstm' in model_name.lower():
        display_name = 'LSTM with Attention'
    elif 'gpt' in model_name.lower():
        display_name = 'Transformer'

    if phase == "pretrain":
        # Create 2x2 subplot for pretrain: Loss vs Epochs, Perplexity vs Epochs, Loss vs Time, Perplexity vs Time
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))

        epochs = range(1, len(metrics['train_loss']) + 1)
        times = metrics.get('epoch_times', epochs)  # Use times if available, else epochs

        # Loss vs Epochs (original)
        ax1.plot(epochs, metrics['train_loss'], 'b-', label='Training Loss', linewidth=2, marker='o')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Loss')
        title_suffix = f' ({total_params:,} params)' if total_params else ''
        ax1.set_title(f'{display_name} Pre-training Loss vs Epochs{title_suffix}')
        ax1.legend()
        ax1.grid(True, alpha=0.3)

        # Perplexity vs Epochs (original)
        ax2.plot(epochs, metrics['perplexity'], 'r-', label='Perplexity', linewidth=2, marker='s')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('Perplexity')
        ax2.set_title(f'{display_name} Pre-training Perplexity vs Epochs{title_suffix}')
        ax2.legend()
        ax2.grid(True, alpha=0.3)

        # Loss vs Time (new)
        ax3.plot(times, metrics['train_loss'], 'b-', label='Training Loss', linewidth=2, marker='o')
        ax3.set_xlabel('Time (seconds)')
        ax3.set_ylabel('Loss')
        ax3.set_title(f'{display_name} Pre-training Loss vs Time{title_suffix}')
        ax3.legend()
        ax3.grid(True, alpha=0.3)

        # Perplexity vs Time (new)
        ax4.plot(times, metrics['perplexity'], 'r-', label='Perplexity', linewidth=2, marker='s')
        ax4.set_xlabel('Time (seconds)')
        ax4.set_ylabel('Perplexity')
        ax4.set_title(f'{display_name} Pre-training Perplexity vs Time{title_suffix}')
        ax4.legend()
        ax4.grid(True, alpha=0.3)

    else:  # finetune
        # Create 2x3 subplot for finetune: Loss vs Epochs, Accuracy vs Epochs, F1 vs Epochs, Loss vs Time, Accuracy vs Time, F1 vs Time
        fig, ((ax1, ax2, ax3), (ax4, ax5, ax6)) = plt.subplots(2, 3, figsize=(18, 10))

        epochs = range(1, len(metrics['train_loss']) + 1)
        times = metrics.get('epoch_times', epochs)  # Use times if available, else epochs
        title_suffix = f' ({total_params:,} params)' if total_params else ''

        # Loss vs Epochs (original)
        ax1.plot(epochs, metrics['train_loss'], 'b-', label='Training Loss', linewidth=2, marker='o')
        ax1.plot(epochs, metrics['val_loss'], 'r-', label='Validation Loss', linewidth=2, marker='s')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Loss')
        ax1.set_title(f'{display_name} Fine-tuning Loss vs Epochs{title_suffix}')
        ax1.legend()
        ax1.grid(True, alpha=0.3)

        # Accuracy vs Epochs (original)
        ax2.plot(epochs, metrics['train_acc'], 'b-', label='Training Accuracy', linewidth=2, marker='o')
        ax2.plot(epochs, metrics['val_acc'], 'r-', label='Validation Accuracy', linewidth=2, marker='s')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('Accuracy')
        ax2.set_title(f'{display_name} Fine-tuning Accuracy vs Epochs{title_suffix}')
        ax2.legend()
        ax2.grid(True, alpha=0.3)

        # F1 Score vs Epochs (original)
        ax3.plot(epochs, metrics['val_f1'], 'g-', label='Validation F1', linewidth=2, marker='^')
        ax3.set_xlabel('Epoch')
        ax3.set_ylabel('F1 Score')
        ax3.set_title(f'{display_name} Fine-tuning F1 vs Epochs{title_suffix}')
        ax3.legend()
        ax3.grid(True, alpha=0.3)

        # Loss vs Time (new)
        ax4.plot(times, metrics['train_loss'], 'b-', label='Training Loss', linewidth=2, marker='o')
        ax4.plot(times, metrics['val_loss'], 'r-', label='Validation Loss', linewidth=2, marker='s')
        ax4.set_xlabel('Time (seconds)')
        ax4.set_ylabel('Loss')
        ax4.set_title(f'{display_name} Fine-tuning Loss vs Time{title_suffix}')
        ax4.legend()
        ax4.grid(True, alpha=0.3)

        # Accuracy vs Time (new)
        ax5.plot(times, metrics['train_acc'], 'b-', label='Training Accuracy', linewidth=2, marker='o')
        ax5.plot(times, metrics['val_acc'], 'r-', label='Validation Accuracy', linewidth=2, marker='s')
        ax5.set_xlabel('Time (seconds)')
        ax5.set_ylabel('Accuracy')
        ax5.set_title(f'{display_name} Fine-tuning Accuracy vs Time{title_suffix}')
        ax5.legend()
        ax5.grid(True, alpha=0.3)

        # F1 Score vs Time (new)
        ax6.plot(times, metrics['val_f1'], 'g-', label='Validation F1', linewidth=2, marker='^')
        ax6.set_xlabel('Time (seconds)')
        ax6.set_ylabel('F1 Score')
        ax6.set_title(f'{display_name} Fine-tuning F1 vs Time{title_suffix}')
        ax6.legend()
        ax6.grid(True, alpha=0.3)

    plt.tight_layout()
    filename = f"{model_name}_{phase}_metrics.png"
    plt.savefig(Config.RESULTS_DIR / filename, dpi=300, bbox_inches='tight')
    plt.close()

## Main training loop

In here is where the main training loop lies.

In [ ]:
# Initialize
config = Config()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

tokenizer = BPETokenizer()

# Determine models to train
models_to_train = ['lstm', 'gpt']
pretrain_epochs = config.PRETRAIN_EPOCHS
finetune_epochs = config.FINETUNE_EPOCHS

for model_name in models_to_train:
    print(f"\n{'='*80}")
    print(f"PROCESSING {model_name.upper()} MODEL")
    print(f"{'='*80}")

    # Create model for pre-training
    model = create_model(model_name, tokenizer, config, for_pretraining=True)
    model = model.to(device)

    trainer = Trainer(model, tokenizer, device, config)

    # Load Wikipedia data
    wiki_texts = load_wikipedia_data(config.WIKI_SUBSET_SIZE)

    # Pre-train
    lr = config.PRETRAIN_LR

    pretrain_metrics = trainer.pretrain(wiki_texts, pretrain_epochs, lr)

    # Plot and save results
    plot_metrics(pretrain_metrics, model_name.upper(), 'pretrain', trainer.total_parameters)

    # Save final checkpoint
    trainer.save_checkpoint(f"{model_name}_pretrained_final.pt", pretrain_epochs)

    # Create model for fine-tuning
    model = create_model(model_name, tokenizer, config, for_pretraining=False)
    model = model.to(device)

    trainer = Trainer(model, tokenizer, device, config)

    # Load from just completed pre-training (skip incompatible layers)
    pretrained_path = config.CHECKPOINT_DIR / f"{model_name}_pretrained_final.pt"
    if pretrained_path.exists():
        trainer.load_checkpoint(str(pretrained_path), strict=False)

        # Load IMDB data
        train_imdb, test_imdb = load_imdb_data()

        train_dataset = IMDBDataset(train_imdb, tokenizer, config.MAX_LEN)
        test_dataset = IMDBDataset(test_imdb, tokenizer, config.MAX_LEN)

        train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True)
        test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE, shuffle=False)

        # Fine-tuning
        lr = config.FINETUNE_LR

        finetune_metrics = trainer.finetune(train_loader, test_loader, finetune_epochs, lr)

        # Final evaluation
        final_results = trainer.evaluate(test_loader)

        print(f"\nFinal {model_name.upper()} Results:")
        print(f"  Accuracy: {final_results['accuracy']:.4f}")
        print(f"  F1 Score: {final_results['f1']:.4f}")
        print(f"  Precision: {final_results['precision']:.4f}")
        print(f"  Recall: {final_results['recall']:.4f}")

        # Plot metrics
        plot_metrics(finetune_metrics, model_name.upper(), 'finetune', trainer.total_parameters)

        # Plot confusion matrix
        plt.figure(figsize=(8, 6))
        cm = confusion_matrix(final_results['labels'], final_results['predictions'])
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=['Negative', 'Positive'],
                    yticklabels=['Negative', 'Positive'])
        display_name = 'LSTM with Attention' if 'lstm' in model_name.lower() else 'Transformer' if 'gpt' in model_name.lower() else model_name.upper()
        plt.title(f'{display_name} Confusion Matrix ({trainer.total_parameters:,} params)')
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')
        plt.savefig(config.RESULTS_DIR / f"{model_name}_confusion_matrix.png", dpi=300, bbox_inches='tight')
        plt.close()

        # Save results (simplified to avoid circular references)
        results_file = config.RESULTS_DIR / f"{model_name}_results.json"
        with open(results_file, 'w') as f:
            # Convert complex objects to simple dictionaries
            simple_results = {}
            for key, value in final_results.items():
                if isinstance(value, (float, int, str)):
                    simple_results[key] = value
                else:
                    simple_results[key] = str(value)

            json.dump({
                'final_metrics': simple_results,
                'model_name': model_name
            }, f, indent=2)

print(f"\n{'='*80}")
print("TRAINING COMPLETE!")
print(f"{'='*80}")
print(f"Checkpoints saved in: {config.CHECKPOINT_DIR}")
print(f"Results saved in: {config.RESULTS_DIR}")
print(f"Logs saved in: {config.LOGS_DIR}")

Using device: cuda
downloading https://openaipublic.blob.core.windows.net/gpt-2/models/124M/encoder.json to /root/.cache/mingpt/encoder.json
downloading https://openaipublic.blob.core.windows.net/gpt-2/models/124M/vocab.bpe to /root/.cache/mingpt/vocab.bpe

PROCESSING LSTM MODEL
LSTM with Attention: number of parameters: 1.62M
Initialized LSTM with Attention with 1,621,218 parameters
Loading Wikipedia dataset (subset of 5 articles)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

20231101.en/train-00000-of-00041.parquet:   0%|          | 0.00/420M [00:00<?, ?B/s]

...

Training Epoch 2/3: 100%|██████████| 6250/6250 [00:50<00:00, 124.40it/s, loss=0.1242, acc=0.9232]


Epoch 2:
  Train Loss: 0.2054, Train Acc: 0.9232
  Val Loss: 0.3481, Val Acc: 0.8654, Val F1: 0.8593
  Time: 66.8s (Total: 127.8s)
Evaluation data saved: results/gpt_finetune_evaluation.json
Checkpoint saved: checkpoints/gpt_finetuned_best.pt
Checkpoint saved: checkpoints/gpt_finetuned_best.pt
Checkpoint saved: checkpoints/gpt_finetuned_epoch_2.pt


Training Epoch 3/3: 100%|██████████| 6250/6250 [00:52<00:00, 119.91it/s, loss=0.2652, acc=0.9574]


Epoch 3:
  Train Loss: 0.1294, Train Acc: 0.9574
  Val Loss: 0.3875, Val Acc: 0.8616, Val F1: 0.8578
  Time: 66.0s (Total: 193.8s)
Evaluation data saved: results/gpt_finetune_evaluation.json
Checkpoint saved: checkpoints/gpt_finetuned_epoch_3.pt
Training summary saved: results/gpt_training_summary.json



Final GPT Results:
  Accuracy: 0.8616
  F1 Score: 0.8578
  Precision: 0.8816
  Recall: 0.8354

TRAINING COMPLETE!
Checkpoints saved in: checkpoints
Results saved in: results
Logs saved in: logs
